<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Bölüm 4: Alıştırma Çözümleri

Bu not defterinde kullanılan paketler:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.9
torch version: 2.9.0
tokenizers version: 0.21.4


&nbsp;
## Alıştırma 4.1: MATH-500 üzerinde düşünce zinciri istemi kullanmak

- Değişiklik yalnızca, istem şablonunu uyguladıktan sonra bir istem soneki eklemeyi gerektiriyor; örneğin `"\n\nExplain step by step."`
- 3. bölümdeki MATH-500 değerlendirme fonksiyonunun değiştirilmiş hâli aşağıda gösterilmiştir (değişiklikler `# NEW` ile yorumlanmıştır)

```python
import json
from pathlib import Path
import time

from reasoning_from_scratch.ch03 import (
    eta_progress_message,
    extract_final_candidate,
    render_prompt,
    grade_answer,
    generate_text_stream_concat,
)


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=512,
    verbose=False,
    prompt_suffix=""  # NEW
):

    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"])
            prompt += prompt_suffix  # NEW
            gen_text = generate_text_stream_concat(
                model, tokenizer, prompt, device,
                max_new_tokens=max_new_tokens,
                verbose=verbose,
            )

            extracted = extract_final_candidate(
                gen_text
            )
            is_correct = grade_answer(
                extracted, row["answer"]
            )
            num_correct += int(is_correct)

            record = {
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            progress_msg = eta_progress_message(
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)
            if verbose:
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    seconds_elapsed = time.time() - start_time
    acc = num_correct / num_examples if num_examples else 0.0
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"Total time: {seconds_elapsed/60:.1f} min")
    print(f"Logs written to: {out_path}")
    return num_correct, num_examples, acc
```

- 3. bölümdeki temel çizgiye göre iyileşmeler aşağıda gösterilmiştir

|    | Yöntem                                       | Model     | Doğruluk | Süre       |
|----|----------------------------------------------|-----------|----------|------------|
| 1  | Temel çizgi (3. bölüm), açgözlü kod çözme    | Temel     | %15.2    | 10.1 dk    |
| 2  | Temel çizgi (3. bölüm), açgözlü kod çözme    | Akıl yür. | %48.2    | 182.1 dk   |
| 3  | Düşünce zinciri istemi ("CoT")               | Temel     | %40.6    | 84.5 dk    |

- Kolaylık olsun diye, [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts) klasöründeki [cot_prompting_math500.py](../02_math500-inference-scaling-scripts/cot_prompting_math500.py) betiğini çalıştırabilirsiniz

&nbsp;
## Alıştırma 4.2: MATH-500 üzerinde sıcaklık ölçeklemesi ve top-p süzgeci kullanmak       

- Burada `generate_text_stream_concat` fonksiyonunu `generate_text_stream_concat_flex` fonksiyonuyla değiştirmemiz ve içine `generate_text_top_p_stream_cache` fonksiyonunu takmamız gerekiyor
- - 3. bölümdeki MATH-500 değerlendirme fonksiyonunun değiştirilmiş hâli aşağıda gösterilmiştir (değişiklikler `# NEW` ile yorumlanmıştır)

```python
import json
from pathlib import Path
import time

from reasoning_from_scratch.ch03 import (
    eta_progress_message,
    extract_final_candidate,
    render_prompt,
    grade_answer,
    generate_text_stream_concat,
)
from reasoning_from_scratch.ch04 import generate_text_stream_concat_flex


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=512,
    verbose=False,
    temperature=1.0,  # NEW
    top_p=1.0,        # NEW
):

    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"])
            gen_text = generate_text_stream_concat_flex( # NEW
                model, tokenizer, prompt, device,
                max_new_tokens=max_new_tokens,
                verbose=verbose,
                generate_func=generate_text_top_p_stream_cache,  # NEW
                temperature=temperature,                         # NEW
                top_p=top_p                                      # NEW
            )

            extracted = extract_final_candidate(
                gen_text
            )
            is_correct = grade_answer(
                extracted, row["answer"]
            )
            num_correct += int(is_correct)

            record = {
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            progress_msg = eta_progress_message(
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)
            if verbose:
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    seconds_elapsed = time.time() - start_time
    acc = num_correct / num_examples if num_examples else 0.0
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"Total time: {seconds_elapsed/60:.1f} min")
    print(f"Logs written to: {out_path}")
    return num_correct, num_examples, acc
```

- Yöntem `temperature` 0.9 ve `top_p` 0.9 ile çalıştırıldığında, aşağıdaki tablodaki temel çizgiye (1. satır) kıyasla yalnızca küçük bir fark oluşuyor; ancak bu beklenen bir durum, çünkü bu yalnızca öz tutarlılık örneklemesi için bir hazırlık

|      | Yöntem                                    | Model     | Doğruluk | Süre      |
| ---- | ----------------------------------------- | --------- | -------- | --------- |
| 1    | Temel çizgi (3. bölüm), açgözlü kod çözme | Temel     | %15.2    | 10.1 dk   |
| ...  | ...                                       | ...       | ...      | ...       |
| 4    | Sıcaklık ve top-p ("Top-p")               | Temel     | %17.8    | 30.7 dk   |

- Kolaylık olsun diye, [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts) klasöründeki [self_consistency_math500.py](../02_math500-inference-scaling-scripts/self_consistency_math500.py) betiğini çalıştırabilirsiniz
- Teknik olarak bu bir öz tutarlılık örnekleme betiğidir; ancak `--num_samples 1` ayarlarsak öz tutarlılık örnekleme kısmı fiilen devre dışı kalır

&nbsp;
## Alıştırma 4.3: MATH-500 üzerinde öz tutarlılık örneklemesi kullanmak

- 3. bölümdeki `evaluate_math500_stream` fonksiyonunu temel alırsak, ilk değişiklik ` gen_text = generate_text_stream_concat(...)` kısmını 4. bölümdeki `results = self_consistency_vote(...)` çağrısıyla değiştirmektir
- İkinci değişiklik, kodun en sık geçen grubun ilk örneğini aldığı basit eşitlik bozma kuralını uygulamayı içerir (ör. 1, 3, 5, 3, 5 sonuçlarımız varsa yanıt olarak 3 döndürülür)
- En sık geçen gruplar `results["majority_winners"]` altında kaydedildiğinden, eşitliği bozmanın bir yolu `results["majority_winners"]` listesinin ilk örneğini, yani `results["majority_winners"][0]` değerini almaktır

```python
import json
from pathlib import Path
import time

from reasoning_from_scratch.ch03 import (
    eta_progress_message,
    render_prompt,
    grade_answer,
)
from reasoning_from_scratch.ch04 import self_consistency_vote


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",    # NEW
    temperature=1.0,     # NEW
    top_p=1.0,           # NEW
    seed=None,           # NEW
    num_samples=10,      # NEW
):

    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"])

            ##############################################################
            # NEW
            prompt += prompt_suffix
            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # resolve ties
            if results["final_answer"] is None:
                extracted = results["majority_winners"][0]
            else:
                extracted = results["final_answer"]

            # extracted = extract_final_candidate(
            #     gen_text
            # )

            # Optionally, get long answer
            if extracted is not None:
                for idx, s in enumerate(results["short_answers"]):
                    if s == extracted:
                        long_answer = results["full_answers"][idx]
                        break
            gen_text = long_answer
            ##############################################################

            is_correct = grade_answer(
                extracted, row["answer"]
            )
            num_correct += int(is_correct)

            record = {
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            progress_msg = eta_progress_message(
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)
            if verbose:
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    seconds_elapsed = time.time() - start_time
    acc = num_correct / num_examples if num_examples else 0.0
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"Total time: {seconds_elapsed/60:.1f} min")
    print(f"Logs written to: {out_path}")
    return num_correct, num_examples, acc
```

- Öz tutarlılık örneklemesi kullanıldığında elde edilen başarım iyileşmeleri aşağıdaki tabloda özetlenmiştir (5-7. satırlar ve 9-12. satırlar)

|      | Yöntem                                    | Model     | Doğruluk | Süre      |
| ---- | ----------------------------------------- | --------- | -------- | --------- |
| 1    | Temel çizgi (3. bölüm), açgözlü kod çözme | Temel     | %15.2    | 10.1 dk   |
| 2    | Temel çizgi (3. bölüm), açgözlü kod çözme | Akıl yür. | %48.2    | 182.1 dk  |
| 3    | Düşünce zinciri istemi ("CoT")            | Temel     | %40.6    | 84.5 dk   |
| 4    | Sıcaklık ve top-p ("Top-p")               | Temel     | %17.8    | 30.7 dk   |
| 5    | "Top-p" + Öz tutarlılık (n=3)             | Temel     | %29.6    | 97.6 dk   |
| 6    | "Top-p" + Öz tutarlılık (n=5)             | Temel     | %27.8    | 116.8 dk  |
| 7    | "Top-p" + Öz tutarlılık (n=10)            | Temel     | %31.6    | 300.4 dk  |
| 8    | "Top-p" + "CoT"                           | Temel     | %33.4    | 129.2 dk  |
| 9    | Öz tutarlılık (n=3) + "Top-p" + "CoT"     | Temel     | %42.2    | 211.6 dk  |
| 10   | Öz tutarlılık (n=5) + "Top-p" + "CoT"     | Temel     | %48.0    | 452.9 dk  |
| 11   | Öz tutarlılık (n=10) + "Top-p" + "CoT"    | Temel     | %52.0    | 862.6 dk  |
| 12   | Öz tutarlılık (n=3) + "Top-p" + "CoT"     | Akıl yür. | %55.2    | 544.4 dk  |

- Kolaylık olsun diye, bunları yeniden üretmek için [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts) klasöründeki [self_consistency_math500.py](../02_math500-inference-scaling-scripts/self_consistency_math500.py) betiğini çalıştırabilirsiniz; [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts) klasörü hangi ayarların kullanılacağına dair daha fazla bilgi içerir

&nbsp;
## Alıştırma 4.4: Öz tutarlılık örneklemesinde erken durdurma

- Erken durdurma kontrolü, verilen yanıtın birden çok kez sayılıp sayılmadığını, daha açık söylemek gerekirse verilen yanıtın sayısının num_samples / 2 değerinden büyük olup olmadığını kontrol eden birkaç satır kod eklenerek uygulanabilir:

```python
if early_stop and counts[short] > num_samples / 2:
    majority_winners = [short]
    final_answer = short
    break
```

- Değiştirilmiş fonksiyonun tamamı aşağıda gösterilmiştir; değişiklikler `# New` ile vurgulanmıştır

```python
import torch
from collections import Counter

from reasoning_from_scratch.ch03 import (
    extract_final_candidate,
)
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache,
)


def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
    early_stop=True,   # NEW
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(
            answer, fallback="number_then_full"
        )
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1
        groups.setdefault(short, []).append(i)

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

        #########################################################
        # NEW
        # Early stop if one answer already meets >= 50% majority
        if early_stop and counts[short] > num_samples / 2:
            majority_winners = [short]
            final_answer = short
            break
        #########################################################

    if final_answer is None:
        mc = counts.most_common()
        if mc:
            top_freq = mc[0][1]
            majority_winners = [s for s, f in mc if f == top_freq]
            final_answer = mc[0][0] if len(majority_winners) == 1 else None

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }
```

- Kolaylık olsun diye, bu değiştirilmiş fonksiyonu MATH-500 veri kümesi üzerinde kullanmak için [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts) klasöründeki [self_consistency_math500.py](../02_math500-inference-scaling-scripts/self_consistency_math500.py) betiğini `--early_stop` bayrağıyla çalıştırabilirsiniz